# 11 税考慮(after-tax)バックテスト — 課税口座 vs NISA

`irp.tax` は日本居住者の **after-tax リターン** を計算する:**年次 mark-to-market** で譲渡益課税(20.315%)を当て、**3年の損失繰越**で相殺、**NISA は非課税**。税率は仮定(`configs/tax_japan.yaml`)でレポートに明示する。投資助言ではない。

In [ ]:
import numpy as np
import pandas as pd
from irp import signals as S, backtest as B, tax as TX, visualization as V
from irp.utils.config import load_config

# Synthetic panel with a persistent edge so the strategy is profitable
# (tax only bites on gains). No network.
rng = np.random.default_rng(11)
idx = pd.bdate_range('2015-01-01', periods=2000)
names = [f'A{i}' for i in range(10)]
edge = np.linspace(-0.0003, 0.0005, 10)
rets = pd.DataFrame(
    {n: rng.normal(edge[i], 0.012, len(idx)) for i, n in enumerate(names)}, index=idx
)
prices = 100 * np.exp(rets.cumsum())
sig = S.momentum_signal(prices, lookback=252, skip=21)
w = B.rebalanced(S.long_short_quantile(sig.lag(1).score, 0.3), 'ME')
strategy = B.run_backtest(w, rets, cost_model=B.CostModel(5, 2), lag=1)
print('pre-tax:', B.summary(strategy, 252)[['ann_return', 'sharpe']].round(3).to_dict())

## 年次の gross vs after-tax(損失繰越込み)

In [ ]:
taxable = TX.TaxConfig.from_config(load_config('tax_japan'))
print(f'rate={taxable.capital_gains_rate:.5f}, loss carryforward={taxable.loss_carryforward_years}y')
TX.annual_after_tax(strategy.returns, taxable).round(4)

## 口座比較:pre-tax vs 課税 vs NISA

In [ ]:
cmp = TX.compare_accounts(strategy.returns, taxable)
print(cmp.round(4))
print(f"\nNISA は非課税なので pre-tax と一致。課税口座の年率ドラッグ = "
      f"{cmp['annual_tax_drag']*100:.2f}%/年、累計納税 = {cmp['total_tax_paid']:.3f}(成長1あたり)")

## after-tax エクイティ:NISA(=非課税)vs 課税口座

In [ ]:
nisa_eq = TX.after_tax_equity(strategy.returns, TX.TaxConfig.nisa())
taxable_eq = TX.after_tax_equity(strategy.returns, taxable)
print('final growth-of-1 — NISA:', round(float(nisa_eq.iloc[-1]), 3),
      '| taxable:', round(float(taxable_eq.iloc[-1]), 3))
V.equity_curves({'NISA (tax-free)': nisa_eq, 'Taxable (20.315%)': taxable_eq},
                title='After-tax growth of 1 (annual)').show()

課税口座は毎年の益に課税され複利が削られる(ドラッグ)。NISA はそれが無いぶん長期で開く。

**正直な注記**: 合成データ・**年次 mark-to-market 仮定**(長期保有では実際の実現課税より保守的=ドラッグ過大)・配当/外国源泉は未モデル(合成に配当が無い、税率は config に用意済)・NISA の年間/生涯枠(¥360万/¥1800万)は return 空間では未モデル。*方法論*の実演で投資結果ではない。